In [1]:
import pandas as pd

data = [
    ["E001","Sales","2026-05-10 09:00:00","2026-05-10 17:30:00",20],
    ["E002","HR","2026-05-10 09:15:00","2026-05-10 16:45:00",15],
    ["E003","IT","2026-05-10 10:30:00","2026-05-10 14:00:00",6],
    ["E004","Sales","2026-05-10 08:45:00","2026-05-10 18:00:00",25],
    ["E005","HR","2026-05-10 11:30:00","2026-05-10 15:00:00",4],
    ["E006","IT","2026-05-10 09:30:00","2026-05-10 19:00:00",30],
    ["E007","Finance","2026-05-10 10:15:00","2026-05-10 13:30:00",3],
    ["E008","Sales","2026-05-10 09:00:00","2026-05-10 17:00:00",18],
    ["E009","Finance","2026-05-10 12:00:00","2026-05-10 16:00:00",5],
    ["E010","IT","2026-05-10 08:30:00","2026-05-10 17:30:00",22],
    ["E011","HR","2026-05-10 10:45:00","2026-05-10 18:15:00",19],
    ["E012","Sales","2026-05-10 11:15:00","2026-05-10 14:30:00",7],
    ["E013","IT","2026-05-10 09:10:00","2026-05-10 18:40:00",28],
    ["E014","Finance","2026-05-10 09:50:00","2026-05-10 17:20:00",16],
    ["E015","HR","2026-05-10 08:40:00","2026-05-10 16:10:00",14]
]

df = pd.DataFrame(data, columns=[
    "employee_id","department","clock_in","clock_out","tasks_completed"
])

df.to_csv("attendance.csv", index=False)

print("CSV file created successfully!")

CSV file created successfully!


In [2]:
import pandas as pd

data = [
    ["E001","T001",5],
    ["E001","T002",7],
    ["E002","T003",4],
    ["E002","T004",6],
    ["E003","T005",2],
    ["E003","T006",4],
    ["E004","T007",10],
    ["E004","T008",8],
    ["E005","T009",1],
    ["E005","T010",3],
    ["E006","T011",12],
    ["E006","T012",9],
    ["E007","T013",1],
    ["E007","T014",2],
    ["E008","T015",6],
    ["E008","T016",7],
    ["E009","T017",2],
    ["E009","T018",3],
    ["E010","T019",11],
    ["E010","T020",10],
    ["E011","T021",8],
    ["E011","T022",7],
    ["E012","T023",3],
    ["E012","T024",4],
    ["E013","T025",14],
    ["E013","T026",12],
    ["E014","T027",9],
    ["E014","T028",6],
    ["E015","T029",5],
    ["E015","T030",6]
]

df = pd.DataFrame(data, columns=["employee_id","task_id","tasks_completed"])

df.to_csv("tasks.csv", index=False)

print("tasks.csv created successfully")

tasks.csv created successfully


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("AttendanceAnalysis") \
    .getOrCreate()

In [5]:
attendance = spark.read.csv(
    "attendance.csv",
    header=True,
    inferSchema=True
)

tasks = spark.read.csv(
    "tasks.csv",
    header=True,
    inferSchema=True
)


In [6]:
combined = attendance.join(
    tasks,
    "employee_id"
)


In [8]:
attendance = attendance.withColumnRenamed("tasks_completed", "tasks_completed_att")

tasks = tasks.withColumnRenamed("tasks_completed", "tasks_completed_task")

combined = attendance.join(tasks, "employee_id")

In [10]:
from pyspark.sql.functions import unix_timestamp

combined = combined.withColumn(
    "work_hours",
    (unix_timestamp("clock_out") - unix_timestamp("clock_in")) / 3600
)

In [11]:
from pyspark.sql.functions import avg

metrics = combined.groupBy("department").agg(
    avg("tasks_completed_att").alias("avg_tasks_attendance"),
    avg("tasks_completed_task").alias("avg_tasks_from_tasks"),
    avg("work_hours").alias("avg_hours")
)

metrics.show()

+----------+--------------------+--------------------+-----------------+
|department|avg_tasks_attendance|avg_tasks_from_tasks|        avg_hours|
+----------+--------------------+--------------------+-----------------+
|     Sales|                17.5|                6.25|             7.25|
|        HR|                13.0|                 5.0|              6.5|
|   Finance|                 8.0|  3.8333333333333335|4.916666666666667|
|        IT|                21.5|                9.25|            7.875|
+----------+--------------------+--------------------+-----------------+



In [12]:
# Save Output
metrics.write.mode("overwrite") \
    .csv("department_kpi")
